In [ ]:
import json, requests, time, os
from collections import Counter
from google.colab import drive, userdata

drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive'
TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

def load_results(path):
    with open(path) as f:
        return {r['id']: r for r in json.load(f)}

llama_m = load_results(f'{PROJECT_DIR}/medmcqa_llama_results.json')
qwen_m = load_results(f'{PROJECT_DIR}/medmcqa_qwen_results.json')
gemma_m = load_results(f'{PROJECT_DIR}/medmcqa_gemma3n_results.json')

with open(f'{PROJECT_DIR}/medmcqa_sample.json') as f:
    questions = json.load(f)

unanimous_questions = []
for q in questions:
    qid = q['id']
    preds = [llama_m[qid]['pred'], qwen_m[qid]['pred'], gemma_m[qid]['pred']]
    correct = [llama_m[qid]['correct'], qwen_m[qid]['correct'], gemma_m[qid]['correct']]
    if any(p is None for p in preds):
        continue
    if all(c == 0 for c in correct) and len(set(preds)) == 1:
        unanimous_questions.append({
            'id': qid,
            'question': q['question'],
            'options': q['options'],
            'gold': q['answer_idx'],
            'modal_wrong': preds[0],
            'subject': q['subject'],
        })

print(f"Project dir: {PROJECT_DIR}")
print(f"Together API key loaded: {bool(TOGETHER_API_KEY)}")
print(f"MedMCQA questions loaded: {len(questions)}")
print(f"Unanimous-wrong questions identified: {len(unanimous_questions)}")
print(f"(Should be ~227 based on Cell 8a)\n")

for q in unanimous_questions[:2]:
    print(f"  Subject: {q['subject']}")
    print(f"  Q: {q['question'][:120]}...")
    print(f"  All 3 picked: {q['modal_wrong']}  |  Gold: {q['gold']}")
    print()

In [ ]:
def classify_bias(question, options, gold, wrong_answer, api_key):
    gold_text = options.get(gold, '')
    wrong_text = options.get(wrong_answer, '')
    opts_text = "\n".join([f"{k}: {v}" for k, v in options.items()])
    prompt = f"""You are an expert medical educator. Classify the cognitive bias that caused this AI model error.

Question: {question}

Options:
{opts_text}

Correct answer: {gold} - {gold_text}
Wrong answer chosen: {wrong_answer} - {wrong_text}

Reply with ONLY one of these labels:
- AVAILABILITY_BIAS (over-weighting salient/memorable symptoms)
- ANCHORING_BIAS (over-relying on first piece of information)
- FRAMING_EFFECT (different conclusion from same info presented differently)
- PREMATURE_CLOSURE (stopping reasoning too early)
- OTHER

Single label only, no explanation:"""

    for attempt in range(3):
        try:
            r = requests.post(
                "https://api.together.xyz/v1/chat/completions",
                headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
                json={"model": "meta-llama/Llama-3.3-70B-Instruct-Turbo",
                      "messages": [{"role": "user", "content": prompt}],
                      "max_tokens": 10, "temperature": 0.0},
                timeout=30
            )
            if r.status_code == 503:
                time.sleep(3 * (attempt + 1))
                continue
            if r.status_code != 200:
                return 'ERROR'
            text = r.json()['choices'][0]['message']['content'].strip().upper()
            for label in ['AVAILABILITY_BIAS', 'ANCHORING_BIAS', 'FRAMING_EFFECT',
                          'PREMATURE_CLOSURE', 'OTHER']:
                if label in text:
                    return label
            return 'OTHER'
        except Exception:
            if attempt == 2:
                return 'ERROR'
            time.sleep(2)
    return 'ERROR'

print(f"Classifying {len(unanimous_questions)} MedMCQA unanimous-wrong questions...")
print(f"Model: Llama-3.3-70B-Instruct-Turbo (identical to MedQA bias work)\n")

medmcqa_unanim_labels = []
errors = 0
for i, q in enumerate(unanimous_questions):
    label = classify_bias(q['question'], q['options'], q['gold'], q['modal_wrong'], TOGETHER_API_KEY)
    if label == 'ERROR':
        errors += 1
    medmcqa_unanim_labels.append({
        'id': q['id'],
        'bias_type': label,
        'modal_wrong': q['modal_wrong'],
        'gold': q['gold'],
        'subject': q['subject'],
    })
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(unanimous_questions)} done... ({errors} errors so far)")
    time.sleep(0.3)

with open(f'{PROJECT_DIR}/medmcqa_unanimous_wrong_bias_labels.json', 'w') as f:
    json.dump(medmcqa_unanim_labels, f)
print(f"\nSaved to {PROJECT_DIR}/medmcqa_unanimous_wrong_bias_labels.json")

valid = [b for b in medmcqa_unanim_labels if b['bias_type'] != 'ERROR']
counts = Counter(b['bias_type'] for b in valid)
total = len(valid)
print(f"\n=== MedMCQA Unanimous-Wrong Bias Distribution (n={total}, errors={errors}) ===")
for bias, count in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"  {bias}: {count} ({count/total*100:.1f}%)")

print(f"\n=== Cross-dataset comparison ===")
medqa_unanim = {'PREMATURE_CLOSURE': 29, 'ANCHORING_BIAS': 18, 'OTHER': 1, 'AVAILABILITY_BIAS': 0}
mq_total = 48
print(f"{'Category':<22} {'MedQA (n=48)':<18} {'MedMCQA (n='+str(total)+')':<20}")
print("-" * 60)
for cat in ['PREMATURE_CLOSURE', 'ANCHORING_BIAS', 'AVAILABILITY_BIAS', 'OTHER']:
    mq = medqa_unanim.get(cat, 0)
    mm = counts.get(cat, 0)
    mq_pct = mq / mq_total * 100
    mm_pct = mm / total * 100 if total > 0 else 0
    print(f"  {cat:<22} {mq:>3} ({mq_pct:>5.1f}%)     {mm:>3} ({mm_pct:>5.1f}%)")

from scipy import stats
medqa_arr = [29, 18, 1]
medmcqa_arr = [counts.get('PREMATURE_CLOSURE', 0),
               counts.get('ANCHORING_BIAS', 0),
               counts.get('AVAILABILITY_BIAS', 0) + counts.get('OTHER', 0)]

if sum(medmcqa_arr) > 0:
    chi2, p, dof, _ = stats.chi2_contingency([medqa_arr, medmcqa_arr])
    print(f"\nChi-square (MedQA unanimous vs MedMCQA unanimous):")
    print(f"  chi2 = {chi2:.3f}  dof = {dof}  p = {p:.4f}")
    if p > 0.05:
        print(f"  → Distributions NOT significantly different — bias profile replicates across datasets")
    else:
        print(f"  → Distributions significantly different — dataset-dependent failure modes")

print(f"\n=== Unanimous-wrong by medical subject ===")
subj_counts = Counter(b['subject'] for b in valid)
for subj, count in subj_counts.most_common(10):
    print(f"  {subj:<35s} {count:>3} ({count/total*100:.1f}%)")

In [ ]:
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

def classify_bias_gpt(question, options, gold, wrong_answer, api_key):
    gold_text = options.get(gold, '')
    wrong_text = options.get(wrong_answer, '')
    opts_text = "\n".join([f"{k}: {v}" for k, v in options.items()])
    prompt = f"""You are an expert medical educator. Classify the cognitive bias that caused this AI model error.

Question: {question}

Options:
{opts_text}

Correct answer: {gold} - {gold_text}
Wrong answer chosen: {wrong_answer} - {wrong_text}

Reply with ONLY one of these labels:
- AVAILABILITY_BIAS (over-weighting salient/memorable symptoms)
- ANCHORING_BIAS (over-relying on first piece of information)
- FRAMING_EFFECT (different conclusion from same info presented differently)
- PREMATURE_CLOSURE (stopping reasoning too early)
- OTHER

Single label only, no explanation:"""
    try:
        r = requests.post(
            "https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
            json={"model": "gpt-4o-mini",
                  "messages": [{"role": "user", "content": prompt}],
                  "max_tokens": 10, "temperature": 0.0},
            timeout=30
        )
        if r.status_code != 200:
            return 'ERROR'
        text = r.json()['choices'][0]['message']['content'].strip().upper()
        for label in ['AVAILABILITY_BIAS', 'ANCHORING_BIAS', 'FRAMING_EFFECT',
                      'PREMATURE_CLOSURE', 'OTHER']:
            if label in text:
                return label
        return 'OTHER'
    except:
        return 'ERROR'

with open(f'{PROJECT_DIR}/medmcqa_unanimous_wrong_bias_labels.json') as f:
    llama_labels = {b['id']: b for b in json.load(f)}

print(f"Classifying 227 MedMCQA unanimous-wrong with GPT-4o-mini...\n")

gpt_labels = []
errors = 0
for i, q in enumerate(unanimous_questions):
    label = classify_bias_gpt(q['question'], q['options'], q['gold'],
                              q['modal_wrong'], OPENAI_API_KEY)
    if label == 'ERROR':
        errors += 1
    gpt_labels.append({
        'id': q['id'],
        'bias_type_gpt': label,
        'bias_type_llama70b': llama_labels[q['id']]['bias_type'],
        'modal_wrong': q['modal_wrong'],
        'gold': q['gold'],
        'subject': q['subject'],
    })
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(unanimous_questions)} done... ({errors} errors)")
    time.sleep(0.2)

with open(f'{PROJECT_DIR}/medmcqa_unanimous_wrong_bias_gpt4omini.json', 'w') as f:
    json.dump(gpt_labels, f)

valid = [b for b in gpt_labels if b['bias_type_gpt'] != 'ERROR']
gpt_counts = Counter(b['bias_type_gpt'] for b in valid)
total = len(valid)

print(f"\n=== Three-way comparison: PC vs Anchoring across datasets and classifiers ===")
print(f"{'':22s} {'PC':>20s} {'ANCHOR':>12s}")
print("-" * 60)

print(f"{'MedQA / Llama-70B':22s} {'29 (60.4%)':>20s} {'18 (37.5%)':>12s}")

print(f"{'MedQA / GPT-4o-mini':22s} {'41 (85.4%)':>20s} {'7 (14.6%)':>12s}")

print(f"{'MedMCQA / Llama-70B':22s} {'176 (77.5%)':>20s} {'34 (15.0%)':>12s}")

pc_g = gpt_counts.get('PREMATURE_CLOSURE', 0)
an_g = gpt_counts.get('ANCHORING_BIAS', 0)
print(f"{'MedMCQA / GPT-4o-mini':22s} {pc_g} ({pc_g/total*100:.1f}%)"
      .ljust(43) + f"{an_g} ({an_g/total*100:.1f}%)".rjust(15))

print(f"\n=== Diagnosis ===")
print(f"  If MedMCQA-GPT looks like MedMCQA-Llama (high PC, low anchor):")
print(f"     → Real dataset effect; bias-elevation patterns are dataset-dependent")
print(f"  If MedMCQA-GPT looks like MedQA-GPT (already classifier-shifted):")
print(f"     → Llama-70B is just less stable on PC-vs-anchoring boundary;")
print(f"       the dataset shift is a classifier-instability artifact")

from sklearn.metrics import cohen_kappa_score
pairs = [(b['bias_type_llama70b'], b['bias_type_gpt'])
         for b in valid if b['bias_type_llama70b'] != 'ERROR']
llama_arr = [p[0] for p in pairs]
gpt_arr = [p[1] for p in pairs]
agree = sum(1 for l, g in pairs if l == g)
print(f"\nInter-classifier agreement on MedMCQA unanimous-wrong:")
print(f"  Raw agreement: {agree}/{len(pairs)} = {agree/len(pairs)*100:.1f}%")
print(f"  Cohen's kappa: {cohen_kappa_score(llama_arr, gpt_arr):.3f}")
print(f"  (MedQA equivalent kappa was 0.181)")

In [ ]:
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

def classify_bias_gpt(question, options, gold, wrong_answer, api_key):
    gold_text = options.get(gold, '')
    wrong_text = options.get(wrong_answer, '')
    opts_text = "\n".join([f"{k}: {v}" for k, v in options.items()])
    prompt = f"""You are an expert medical educator. Classify the cognitive bias that caused this AI model error.

Question: {question}

Options:
{opts_text}

Correct answer: {gold} - {gold_text}
Wrong answer chosen: {wrong_answer} - {wrong_text}

Reply with ONLY one of these labels:
- AVAILABILITY_BIAS (over-weighting salient/memorable symptoms)
- ANCHORING_BIAS (over-relying on first piece of information)
- FRAMING_EFFECT (different conclusion from same info presented differently)
- PREMATURE_CLOSURE (stopping reasoning too early)
- OTHER

Single label only, no explanation:"""
    try:
        r = requests.post(
            "https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
            json={"model": "gpt-4o-mini",
                  "messages": [{"role": "user", "content": prompt}],
                  "max_tokens": 10, "temperature": 0.0},
            timeout=30
        )
        if r.status_code != 200:
            return 'ERROR'
        text = r.json()['choices'][0]['message']['content'].strip().upper()
        for label in ['AVAILABILITY_BIAS', 'ANCHORING_BIAS', 'FRAMING_EFFECT',
                      'PREMATURE_CLOSURE', 'OTHER']:
            if label in text:
                return label
        return 'OTHER'
    except:
        return 'ERROR'

with open(f'{PROJECT_DIR}/medmcqa_unanimous_wrong_bias_labels.json') as f:
    llama_labels = {b['id']: b for b in json.load(f)}

print(f"Classifying 227 MedMCQA unanimous-wrong with GPT-4o-mini...\n")

gpt_labels = []
errors = 0
for i, q in enumerate(unanimous_questions):
    label = classify_bias_gpt(q['question'], q['options'], q['gold'],
                              q['modal_wrong'], OPENAI_API_KEY)
    if label == 'ERROR':
        errors += 1
    gpt_labels.append({
        'id': q['id'],
        'bias_type_gpt': label,
        'bias_type_llama70b': llama_labels[q['id']]['bias_type'],
        'modal_wrong': q['modal_wrong'],
        'gold': q['gold'],
        'subject': q['subject'],
    })
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{len(unanimous_questions)} done... ({errors} errors)")
    time.sleep(0.2)

with open(f'{PROJECT_DIR}/medmcqa_unanimous_wrong_bias_gpt4omini.json', 'w') as f:
    json.dump(gpt_labels, f)

valid = [b for b in gpt_labels if b['bias_type_gpt'] != 'ERROR']
gpt_counts = Counter(b['bias_type_gpt'] for b in valid)
total = len(valid)

print(f"\n=== Three-way comparison: PC vs Anchoring across datasets and classifiers ===")
print(f"{'':22s} {'PC':>20s} {'ANCHOR':>12s}")
print("-" * 60)

print(f"{'MedQA / Llama-70B':22s} {'29 (60.4%)':>20s} {'18 (37.5%)':>12s}")

print(f"{'MedQA / GPT-4o-mini':22s} {'41 (85.4%)':>20s} {'7 (14.6%)':>12s}")

print(f"{'MedMCQA / Llama-70B':22s} {'176 (77.5%)':>20s} {'34 (15.0%)':>12s}")

pc_g = gpt_counts.get('PREMATURE_CLOSURE', 0)
an_g = gpt_counts.get('ANCHORING_BIAS', 0)
print(f"{'MedMCQA / GPT-4o-mini':22s} {pc_g} ({pc_g/total*100:.1f}%)"
      .ljust(43) + f"{an_g} ({an_g/total*100:.1f}%)".rjust(15))

print(f"\n=== Diagnosis ===")
print(f"  If MedMCQA-GPT looks like MedMCQA-Llama (high PC, low anchor):")
print(f"     → Real dataset effect; bias-elevation patterns are dataset-dependent")
print(f"  If MedMCQA-GPT looks like MedQA-GPT (already classifier-shifted):")
print(f"     → Llama-70B is just less stable on PC-vs-anchoring boundary;")
print(f"       the dataset shift is a classifier-instability artifact")

from sklearn.metrics import cohen_kappa_score
pairs = [(b['bias_type_llama70b'], b['bias_type_gpt'])
         for b in valid if b['bias_type_llama70b'] != 'ERROR']
llama_arr = [p[0] for p in pairs]
gpt_arr = [p[1] for p in pairs]
agree = sum(1 for l, g in pairs if l == g)
print(f"\nInter-classifier agreement on MedMCQA unanimous-wrong:")
print(f"  Raw agreement: {agree}/{len(pairs)} = {agree/len(pairs)*100:.1f}%")
print(f"  Cohen's kappa: {cohen_kappa_score(llama_arr, gpt_arr):.3f}")
print(f"  (MedQA equivalent kappa was 0.181)")

In [ ]:
import json
from datetime import datetime

PROJECT_DIR = '/content/drive/MyDrive'

triangulation_summary = {
    'timestamp': datetime.now().isoformat(),
    'session': 'MedMCQA bias analysis + cross-classifier triangulation',

    'four_corner_distribution': {
        'medqa_llama70b':    {'PC': 0.604, 'ANCHOR': 0.375, 'n': 48},
        'medqa_gpt4omini':   {'PC': 0.854, 'ANCHOR': 0.146, 'n': 48},
        'medmcqa_llama70b':  {'PC': 0.775, 'ANCHOR': 0.150, 'n': 227},
        'medmcqa_gpt4omini': {'PC': 0.722, 'ANCHOR': 0.211, 'n': 227},
    },

    'cross_classifier_agreement': {
        'medqa_kappa': 0.181,
        'medmcqa_kappa': 0.215,
        'medmcqa_raw_agreement': 0.683,
        'interpretation': 'Both weak; classifiers cannot reliably distinguish PC from anchoring'
    },

    'robust_findings': [
        'PC dominance: 72-85% across all 4 corners (dataset x classifier)',
        'Unanimous-wrong convergence: 33.6% MedQA, 36.4% MedMCQA',
        'Pairwise lifts replicate: 1.33-1.74x strong-strong, ~1.0x Flan-strong',
        'No expertise reversal: Flan does NOT beat baseline on shared-fail subset (both datasets)',
    ],

    'retracted_or_weakened': [
        'Anchoring elevation in MedQA unanimous-wrong (Llama-70B artifact, retracted)',
        'Dataset-specific anchoring difference (classifier-dependent, weakened)',
    ],

    'methodological_finding': 'LLM-as-judge produces stable MARGINAL distributions on PC dominance but UNRELIABLE per-instance discrimination between PC and anchoring categories.',

    'paper_central_claim': 'Capable LLMs from different families converge on identical wrong distractors in clinical multiple-choice questions at ~9x chance rate, replicating across MedQA-USMLE and MedMCQA. Premature closure is the dominant failure characterization.',

    'next_steps': [
        'DeepSeek-V3 MedMCQA evaluation when Together recovers (~$1.33)',
        'Qualitative typology of MedMCQA unanimous-wrong (5-10 examples)',
        'Paper outline + Section 1 draft',
        'Figures: pairwise lift heatmap, unanimous-wrong rates, PC dominance across 4 corners',
    ]
}

with open(f'{PROJECT_DIR}/triangulation_summary.json', 'w') as f:
    json.dump(triangulation_summary, f, indent=2)

print("Saved triangulation_summary.json")
print(f"\nRobust findings: {len(triangulation_summary['robust_findings'])}")
print(f"Retracted/weakened: {len(triangulation_summary['retracted_or_weakened'])}")
print(f"\nCentral claim: {triangulation_summary['paper_central_claim']}")

In [ ]:
import json, random, requests
from collections import defaultdict
from google.colab import userdata

PROJECT_DIR = '/content/drive/MyDrive'
TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')

print("=== DeepSeek-V3 availability check ===")
try:
    r = requests.post(
        "https://api.together.xyz/v1/chat/completions",
        headers={"Authorization": f"Bearer {TOGETHER_API_KEY}", "Content-Type": "application/json"},
        json={"model": "deepseek-ai/DeepSeek-V3",
              "messages": [{"role": "user", "content": "PING"}],
              "max_tokens": 5, "temperature": 0.0},
        timeout=10
    )
    if r.status_code == 200:
        print(f"  ✓ DeepSeek-V3 is BACK — can run MedMCQA tomorrow")
    else:
        print(f"  ✗ Still down: HTTP {r.status_code}")
except Exception as e:
    print(f"  ✗ Error: {e}")

with open(f'{PROJECT_DIR}/medmcqa_sample.json') as f:
    questions_by_id = {q['id']: q for q in json.load(f)}
with open(f'{PROJECT_DIR}/medmcqa_unanimous_wrong_bias_labels.json') as f:
    bias_labels = {b['id']: b for b in json.load(f)}
with open(f'{PROJECT_DIR}/medmcqa_llama_results.json') as f:
    llama_m = {r['id']: r for r in json.load(f)}
with open(f'{PROJECT_DIR}/medmcqa_qwen_results.json') as f:
    qwen_m = {r['id']: r for r in json.load(f)}
with open(f'{PROJECT_DIR}/medmcqa_gemma3n_results.json') as f:
    gemma_m = {r['id']: r for r in json.load(f)}

candidates = []
for qid, bl in bias_labels.items():
    q = questions_by_id[qid]
    candidates.append({
        'id': qid,
        'question': q['question'],
        'options': q['options'],
        'gold': q['answer_idx'],
        'modal_wrong': bl['modal_wrong'],
        'bias': bl['bias_type'],
        'subject': bl['subject'],
        'q_length': len(q['question']),
    })

print(f"\n=== Sampling diverse examples ===")
print(f"Total unanimous-wrong: {len(candidates)}")

by_subject_bias = defaultdict(list)
for c in candidates:
    by_subject_bias[(c['subject'], c['bias'])].append(c)

random.seed(42)
selected = []
target_subjects = ['Surgery', 'Medicine', 'Pathology', 'Pharmacology',
                   'Gynaecology & Obstetrics', 'Microbiology', 'Anatomy',
                   'Physiology', 'Pediatrics', 'Dental']

for subj in target_subjects:

    for bias in ['PREMATURE_CLOSURE', 'ANCHORING_BIAS']:
        pool = by_subject_bias.get((subj, bias), [])

        pool = sorted(pool, key=lambda x: -x['q_length'])
        if pool:
            selected.append(pool[0])
            break
    if len(selected) >= 10:
        break

print(f"Selected {len(selected)} examples across subjects\n")

for i, c in enumerate(selected, 1):
    print("=" * 75)
    print(f"EXAMPLE {i}  |  Subject: {c['subject']}  |  Bias: {c['bias']}")
    print("=" * 75)
    print(f"\nQuestion:")
    print(c['question'])
    print(f"\nOptions:")
    for letter in 'ABCD':
        text = c['options'][letter]
        marker = ""
        if letter == c['gold']:
            marker = "  ← GOLD"
        if letter == c['modal_wrong']:
            marker += "  ← ALL 3 MODELS PICKED"
        print(f"  {letter}: {text}{marker}")
    print(f"\nBias classification: {c['bias']}")
    print()

In [ ]:
import json

PROJECT_DIR = '/content/drive/MyDrive'

paper_examples = {
    'medmcqa_qualitative_picks': [
        {'example_num': 1, 'subject': 'Surgery', 'theme': 'Obvious treatment, missed contraindication',
         'note': 'Facial abscess + orbital signs → aspiration first, not I&D'},
        {'example_num': 2, 'subject': 'Medicine', 'theme': 'Lipid pattern triggers combo therapy reflex',
         'note': 'Post-MI guidelines = high-intensity statin alone, not statin+fibrate'},
        {'example_num': 3, 'subject': 'Pathology', 'theme': 'Generic next-step over syndrome-specific',
         'note': 'CML signature on DLC → Philadelphia chromosome, not generic BM biopsy'},
        {'example_num': 5, 'subject': 'Obstetrics', 'theme': 'Prototype condition crowds out specific dx',
         'note': 'Placenta accreta is famous; placenta succenturiata is the actual answer'},
        {'example_num': 9, 'subject': 'Pediatrics', 'theme': 'Lay-intuitive answer over technical concept',
         'note': 'Delta bilirubin explains persistent elevation; "normal lowering takes time" is the trap'},
    ],
    'parallel_to_medqa': [
        'idx=0 carpal tunnel ethics',
        'idx=6 DM+PAD flank pain (RAS trap)',
        'idx=23 post-stroke pneumonia (Strep vs Staph)',
        'idx=59 eating disorder dental',
        'idx=93 Down syndrome vs RA cervical instability',
    ],
    'shared_pattern': 'Across both datasets, 4 LLM families converge on the textbook-prototype answer rather than the evidence-warranted answer. This is the qualitative signature of the convergent failure mode.'
}

with open(f'{PROJECT_DIR}/paper_qualitative_examples.json', 'w') as f:
    json.dump(paper_examples, f, indent=2)

print(f"Saved paper_qualitative_examples.json")
print(f"\nCross-dataset pattern: {paper_examples['shared_pattern']}")

In [ ]:
import requests
from google.colab import userdata

TOGETHER_API_KEY = userdata.get('TOGETHER_API_KEY')
r = requests.post(
    "https://api.together.xyz/v1/chat/completions",
    headers={"Authorization": f"Bearer {TOGETHER_API_KEY}", "Content-Type": "application/json"},
    json={"model": "deepseek-ai/DeepSeek-V3",
          "messages": [{"role": "user", "content": "PING"}],
          "max_tokens": 5, "temperature": 0.0},
    timeout=15
)
print(f"DeepSeek status: {r.status_code}")
print(f"Response: {r.text[:200]}")